In [18]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", None)   # don't cut off Arabic text
RAW = Path("../data/raw")                      # relative to notebooks/
CORPUS = RAW / "Banking77_full_corpus.csv"


In [19]:
with open(CORPUS, "r", encoding="utf-8") as f:
    ls = f.readlines()
    
ls[856]

'5,"extra charge on statement","رسوم إضافية على كشف الحساب",856,"How come there is a $1 extra charge on my statement?",Tr6726,"كيف يكون هناك رسوم إضافية بقيمة 1 دولار على كشف الحساب الخاص بي؟",NULL,NULL,Tr17547,"كيف في رسوم اضافية على حسابي قيمتها 1 دولار؟",D2021,"كيف يعني فيه على حسابي رسوم زيادة قيمتها 1 دولار؟"\n'

In [20]:
len(ls[856].split(","))

13

In [21]:
ls[855].split(",")

['5',
 '"extra charge on statement"',
 '"رسوم إضافية على كشف الحساب"',
 '855',
 '"I\'m pretty frustrated about a pound charge that keeps showing up in my account statement. It\'s listed in the app as "pending"',
 ' it never changes',
 ' and now I\'m thinking someone\'s hacked my account!"',
 'Tr6660',
 '"أشعر بالإحباط الشديد بشأن رسوم الجنيه التي تستمر في الظهور في بيان حسابي. إنه مدرج في التطبيق على أنه "معلق" ، ولا يتغير أبدا ، والآن أفكر أن شخصا ما قد اخترق حسابي!"',
 'NULL',
 'NULL',
 'Tr17580',
 '"أنا محبط كثير بسبب رسوم الجنيه الي بتضلها تبين على حسابي. مبين في التطبيق انه معلق وما بتغير، وهلا قاعد بفكرأنه حدا هكر حسابي!"',
 'NULL',
 'NULL\n']

In [22]:
ls[854].split(",")

['5',
 '"extra charge on statement"',
 '"رسوم إضافية على كشف الحساب"',
 '854',
 '"I wanted to know about a pending dollar that is on my statement."',
 'D777',
 '"أردت أن أعرف عن الدولار المعلق الموجود في بياني."',
 'NULL',
 'NULL',
 'Tr17647',
 '"كان بدي أعرف عن الدولار المعلق الموجود في البيان تبعي"',
 'NULL',
 'NULL\n']

In [23]:
ls[0].split(",")

['Intent_ID',
 'Intent_en',
 'Intent_ar',
 'QID',
 'Question_en',
 'QuestionID_MSA1',
 'Question_MSA1',
 'QuestionID_MSA2',
 'Question_MSA2',
 'QuestionID_PAL1',
 'Question_PAL1',
 'QuestionID_PAL2',
 'Question_PAL2\n']

- some rows can't be parsed so they have different number of columns so we will have to drop them, if they aren't too many

In [24]:
import csv

# 1. Read the file with Python's csv module and find the rows that don't
#    split into the same number of fields as the header (13)
with open(CORPUS, encoding="utf-8", newline="") as f:
    records = list(csv.reader(f))

header = records[0]
bad_idx = [i for i, row in enumerate(records) if len(row) != len(header)]

# 2. Record what we drop, so it's never silent
dropped = pd.DataFrame({
    "line": [i + 1 for i in bad_idx],          # 1-based, as your editor shows it
    "QID": [records[i][3] for i in bad_idx],
    "intent": [records[i][1] for i in bad_idx],
    "n_fields": [len(records[i]) for i in bad_idx],
})
display(dropped)

# 3. Load everything except those lines
df = pd.read_csv(CORPUS, skiprows=bad_idx)


print(f"loaded {len(df):,} rows, dropped {len(bad_idx)}")

,line,QID,intent,n_fields
0,856,855,extra charge on statement,15
1,931,930,extra charge on statement,14
2,2737,2736,pending top up,15
3,6708,6707,pending card payment,14
4,6713,6712,pending card payment,14
5,8272,8271,transaction charged twice,14
6,9467,9466,transfer fee charged,14
7,9484,9483,transfer fee charged,14


loaded 13,075 rows, dropped 8


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13075 entries, 0 to 13074
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Intent_ID        13075 non-null  int64
 1   Intent_en        13075 non-null  str  
 2   Intent_ar        13075 non-null  str  
 3   QID              13075 non-null  int64
 4   Question_en      13075 non-null  str  
 5   QuestionID_MSA1  13075 non-null  str  
 6   Question_MSA1    13075 non-null  str  
 7   QuestionID_MSA2  2452 non-null   str  
 8   Question_MSA2    2452 non-null   str  
 9   QuestionID_PAL1  13075 non-null  str  
 10  Question_PAL1    13075 non-null  str  
 11  QuestionID_PAL2  2783 non-null   str  
 12  Question_PAL2    2783 non-null   str  
dtypes: int64(2), str(11)
memory usage: 5.5 MB


In [26]:
df.head().T

,0,1,2,3,4
Intent_ID,1,1,1,1,1
Intent_en,card arrival,card arrival,card arrival,card arrival,card arrival
Intent_ar,وصول البطاقة,وصول البطاقة,وصول البطاقة,وصول البطاقة,وصول البطاقة
QID,1,2,3,4,5
Question_en,I am still waiting on my card?,What can I do if my card still hasn't arrived after 2 weeks?,I have been waiting over a week. Is the card still coming?,Can I track my card while it is in the process of delivery?,"How do I know if I will get my card, or if it is lost?"
QuestionID_MSA1,Tr10625,Tr10645,Tr10557,Tr10724,Tr10591
Question_MSA1,ما زلت أنتظر بطاقتي؟,ماذا أفعل إذا لم تصل بطاقتي بعد أسبوعين؟,انا أنتظر منذ أكثر من أسبوع. هل ما زالت البطاقة قادمة؟,هل يمكنني تتبع بطاقتي أثناء عملية التسليم؟,كيف أعرف إذا ما كنت سأحصل على بطاقتي ، أو إذا ضاعت؟
QuestionID_MSA2,NaN,Tr10644,NaN,Tr10562,Tr10677
Question_MSA2,NaN,ماذا علي أن أفعل في حال عدم وصول بطاقتي خلال أسبوعين؟,NaN,أبإمكاني تتبع بطاقتي وهي في عملية التسليم؟,كيف يمكنني أن أعرف ما إن كنت سأحصل على بطاقة أم ضاعت؟
QuestionID_PAL1,D2445,Tr21362,Tr21479,Tr21518,Tr21433


In [27]:
df.QID.nunique()

13075

In [28]:
df[["Question_MSA1", "Question_MSA2", "Question_PAL1", "Question_PAL2"]].notna().sum()

Question_MSA1    13075
Question_MSA2     2452
Question_PAL1    13075
Question_PAL2     2783
dtype: int64

## Why `MSA2` and `PAL2` are mostly empty

- One row is **one English source question** (`QID`), with its Arabic versions side by side in 4 slots: `MSA1`, `MSA2`, `PAL1`, `PAL2`.
- Every question has `MSA1` and `PAL1`. Only some got a second paraphrase: `MSA2` for 2,452 questions, `PAL2` for 2,783. The other slots hold the literal text `NULL`, which pandas reads as `NaN` by default.
- An ID and its text are always empty together (checked below), so these are **empty slots, not missing data**: there is nothing to fill in.
- Each **version** has its own ID, and its prefix gives its split: `Tr` = train, `D` = dev (validation), `Te` = test. The split is set per version, not per question.

In [29]:
SLOTS = ["MSA1", "MSA2", "PAL1", "PAL2"]

# An ID without a text, or a text without an ID, would mean a broken row, not an empty slot
for slot in SLOTS:
    mismatch = df[f"QuestionID_{slot}"].isna() != df[f"Question_{slot}"].isna()
    print(f"{slot}: {mismatch.sum()} rows where only one of ID / text is empty")

has_version = df[[f"Question_{s}" for s in SLOTS]].notna()
has_version.columns = SLOTS
has_version.sum(axis=1).value_counts().sort_index().rename("questions with N versions")

MSA1: 0 rows where only one of ID / text is empty
MSA2: 0 rows where only one of ID / text is empty
PAL1: 0 rows where only one of ID / text is empty
PAL2: 0 rows where only one of ID / text is empty


2    8770
3    3375
4     930
Name: questions with N versions, dtype: int64

## Reshape: one row per Arabic version

- A model trains on (text, label) pairs, so the 4 slots become rows, and the empty slots drop out.
- `qid` stays on every row: it links the versions of one question, and the split in part 4 groups by it.
- `version_id` is kept as it is; its prefix is what part 2 looks at.

In [30]:
parts = []
for slot in SLOTS:
    part = df[["QID", "Intent_ID", "Intent_en", f"QuestionID_{slot}", f"Question_{slot}"]]
    part = part.rename(columns={
        "QID": "qid", "Intent_ID": "intent_id", "Intent_en": "intent",
        f"QuestionID_{slot}": "version_id", f"Question_{slot}": "text",
    })
    part["slot"] = slot
    part["dialect"] = slot[:3]      # "MSA" or "PAL"
    parts.append(part)

versions = (
    pd.concat(parts, ignore_index=True)
    .dropna(subset=["text"])        # an empty slot means no version was written
    .sort_values(["qid", "slot"])
    .reset_index(drop=True)
)[["qid", "intent_id", "intent", "dialect", "slot", "version_id", "text"]]

assert len(versions) == has_version.to_numpy().sum(), "rows lost in the reshape"
print(f"{len(versions):,} Arabic versions from {versions['qid'].nunique():,} questions")
display(versions.head(6))
versions["dialect"].value_counts()

31,385 Arabic versions from 13,075 questions


,qid,intent_id,intent,dialect,slot,version_id,text
0,1,1,card arrival,MSA,MSA1,Tr10625,ما زلت أنتظر بطاقتي؟
1,1,1,card arrival,PAL,PAL1,D2445,بعدني بستنى في البطاقة؟
2,2,1,card arrival,MSA,MSA1,Tr10645,ماذا أفعل إذا لم تصل بطاقتي بعد أسبوعين؟
3,2,1,card arrival,MSA,MSA2,Tr10644,ماذا علي أن أفعل في حال عدم وصول بطاقتي خلال أسبوعين؟
4,2,1,card arrival,PAL,PAL1,Tr21362,ايش بلزم اعمل اذا بطاقتي ما وصلت بعد اسبوعين؟
5,2,1,card arrival,PAL,PAL2,Tr21393,شو اساوي اذا ما وصلت بطاقتي بعد اسبوعين؟


dialect
PAL    15858
MSA    15527
Name: count, dtype: int64

## Do all version IDs look like IDs?

In [31]:
ID_PATTERN = r"(Tr|D|Te)\d+"     # a split prefix, then digits, and nothing else

# First make sure the check can fail: it must accept real IDs and reject anything else
checks = pd.Series({
    "Tr10625": True, "D6": True, "Te409": True,
    "abc": False, "Tr": False, "12": False, "Tr12x": False, "وصول": False,
})
result = pd.Series(checks.index).str.fullmatch(ID_PATTERN)
assert result.tolist() == checks.tolist(), "the pattern accepts or rejects the wrong values"

# fullmatch, not match: match only checks the start, so it would accept "Tr12x"
odd = versions[~versions["version_id"].str.fullmatch(ID_PATTERN)]
print(f"{len(odd)} of {len(versions):,} version IDs don't match")
odd

5 of 31,385 version IDs don't match


,qid,intent_id,intent,dialect,slot,version_id,text
18335,7677,46,declined card payment,PAL,PAL1,Train,"لما بستخدم بطاقتي بتطلعلي رسالة رفض""؟"""
18362,7690,46,declined card payment,PAL,PAL1,Train,"هل كلشي منيح مع بطاقتي ؟ بضل يجيني رسالة رفض"""""
18987,7964,47,pending transfer,PAL,PAL1,Train,"شو يعني إنه التحويل معلق""؟"""
19001,7971,47,pending transfer,PAL,PAL1,Train,"قديه ممكن يضل التحويل معلق""؟"""
21311,8805,53,reverted card payment?,PAL,PAL1,Train,"مرحبا، صار معي مشكلة في الدفع ببطاقتي وبيطلعلي عحسابي انه دفعت. بعدين اتصل فيي البياع بيقلي ما وصلتني مصاري"" ورجع المبلغ لحسابي. بدي أعرف ليش صار هيك."""


## findings

- **8 rows can't be parsed:** an unescaped quote followed by a comma splits them into 14–15 fields. They are dropped and listed above, so 13,075 of the 13,083 questions remain.
- **41 more rows have unescaped quotes** but still parse into the right columns. Their texts keep a stray `"` (e.g. `رفض"؟"`); whether to clean it is decided in part 4.
- **One row = one English source question** with 2–4 Arabic versions: `MSA1` and `PAL1` always, `MSA2` and `PAL2` sometimes. The empty slots are not missing data.
- **Reshaped to one row per version:** 31,385 Arabic versions from 13,075 questions.
- **`version_id` for 5 rows of the full corpus is named differently than the others:** `Train` instead of `Tr<number>`, all in `PAL1`. It is written like that in SinaLab's file (not a parsing error), and all 5 texts appear in the official train file, so they are treated as train.

## does the official validation set leak?

Each version's ID prefix gives its official split. If one version of a question is in train and another in validation, the model is validated on a question it has already seen in another form.

In [32]:
SPLIT_NAMES = {"Tr": "train", "D": "val", "Te": "test"}

# The 5 "Train" IDs are training versions: their texts are in the official train file
prefix = (
    versions["version_id"].str.extract(f"^{ID_PATTERN}$")[0]
    .mask(versions["version_id"] == "Train", "Tr")
)
versions["split"] = prefix.map(SPLIT_NAMES)

assert versions["split"].notna().all(), "a version has no split"
versions["split"].value_counts()

split
train    21547
test      7376
val       2462
Name: count, dtype: int64

In [33]:
# Which splits do the versions of each question fall into?
splits_per_question = versions.groupby("qid")["split"].agg(lambda s: " + ".join(sorted(set(s))))
splits_per_question.value_counts()

split
train          7720
test           3078
train + val    2200
val              77
Name: count, dtype: int64

In [34]:
leaky_qids = splits_per_question[splits_per_question == "train + val"].index
val = versions[versions["split"] == "val"]
leaked = val["qid"].isin(leaky_qids)
print(f"{leaked.sum():,} of {len(val):,} validation versions ({leaked.mean():.0%}) "
      "have a version of the same question in train")

# One example: the same question, one version in train and one in validation
versions[versions["qid"] == leaky_qids[0]]

2,305 of 2,462 validation versions (94%) have a version of the same question in train


,qid,intent_id,intent,dialect,slot,version_id,text,split
0,1,1,card arrival,MSA,MSA1,Tr10625,ما زلت أنتظر بطاقتي؟,train
1,1,1,card arrival,PAL,PAL1,D2445,بعدني بستنى في البطاقة؟,val


## findings

- Each question's versions fall into **train only** (7,720 questions), **test only** (3,078), **validation only** (77), or **train + validation (2,200)**.
- **94% of the official validation versions (2,305 of 2,462) have a version of the same question in train**, a translation or paraphrase of it (e.g. QID 1 above: MSA in train, Palestinian in validation). A score on the official validation set mostly measures questions the model has already seen in another form. How much this inflates the score is measured in step 2 with the baseline.
- **Test never mixes with train or validation**, so the official test files can be used as they are, and our numbers stay comparable with the paper.
- **Decision:** don't use the official validation set. Build our own from the train + validation pool, split by `qid` so all versions of a question land on the same side (part 4).

## Part 4: our own validation split

- **Pool:** every train and validation version in the corpus, minus the 41 versions with a stray `"`. The official test files are kept as published.
- **Grouped by `qid`:** all versions of a question land on the same side, so validation never contains a paraphrase of a training question.
- **Stratified by intent:** each intent keeps about the same share in train and validation.
- `StratifiedGroupKFold` with 10 folds does both; one fold (10%) becomes validation. The seed makes the split reproducible.

In [35]:
SEED = 42
VAL_FRACTION = 0.10

in_pool = versions["split"].isin(["train", "val"])
has_quote = versions["text"].str.contains('"', regex=False)
print(f"dropping {(in_pool & has_quote).sum()} pool versions with a stray quote")

pool = versions[in_pool & ~has_quote].reset_index(drop=True)
print(f"pool: {len(pool):,} versions from {pool['qid'].nunique():,} questions")

dropping 41 pool versions with a stray quote
pool: 23,968 versions from 9,992 questions


In [36]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=round(1 / VAL_FRACTION), shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(pool, y=pool["intent_id"], groups=pool["qid"]))
train_set, val_set = pool.iloc[train_idx], pool.iloc[val_idx]

assert not set(train_set["qid"]) & set(val_set["qid"]), "a question is in both train and val"
assert train_set["intent_id"].nunique() == val_set["intent_id"].nunique() == 77, "an intent is missing"
assert len(train_set) + len(val_set) == len(pool), "rows lost in the split"

print(f"train: {len(train_set):,} versions from {train_set['qid'].nunique():,} questions")
print(f"val:   {len(val_set):,} versions from {val_set['qid'].nunique():,} questions ({len(val_set) / len(pool):.1%})")

train: 21,571 versions from 8,993 questions
val:   2,397 versions from 999 questions (10.0%)


In [37]:
# Is validation a fair sample? Every intent needs enough questions, and the intent mix should match train
val_questions = val_set.groupby("intent")["qid"].nunique()
print(f"validation questions per intent: min {val_questions.min()}, median {val_questions.median():.0f}, max {val_questions.max()}")

share_gap = (train_set["intent"].value_counts(normalize=True) - val_set["intent"].value_counts(normalize=True)).abs().max()
print(f"largest gap in an intent's share between train and val: {share_gap:.2%}")
val_set["dialect"].value_counts(normalize=True)

validation questions per intent: min 4, median 13, max 19
largest gap in an intent's share between train and val: 0.07%


dialect
PAL    0.503129
MSA    0.496871
Name: proportion, dtype: float64

In [38]:
# Part 3, as one check: does any official test text also appear in our train or validation?
TEST_SETS = ["MSA", "PAL", "Saudi", "Moroccan", "Tunisian"]
tests = {name: pd.read_csv(RAW / f"Banking77_Arabized_{name}_test.csv") for name in TEST_SETS}

seen = set(pool["text"].str.strip())
for name, test in tests.items():
    overlap = test["text"].str.strip().isin(seen)
    print(f"{name}: {overlap.sum()} of {len(test):,} test texts also in train/val", test.loc[overlap, "text"].tolist())

MSA: 0 of 3,574 test texts also in train/val []
PAL: 0 of 3,807 test texts also in train/val []
Saudi: 1 of 3,580 test texts also in train/val ['انسرقت بطاقتي']
Moroccan: 2 of 3,574 test texts also in train/val ['تفعيل بطاقتي', 'تفعيل بطاقتي']
Tunisian: 0 of 999 test texts also in train/val []


## Part 4 findings

- **Pool:** 23,968 versions from 9,992 questions, after dropping the 41 versions with a stray quote. 5 questions had no other train/val version and drop out entirely.
- **Our split:** train 21,571 versions (8,993 questions), validation 2,397 versions (999 questions), exactly 10%.
- **The three asserts pass:** no question is on both sides, all 77 intents are in both, and no rows were lost.
- **Validation is a fair sample:** 4–19 questions per intent (median 13), every intent's share within 0.07% of train, and half MSA, half Palestinian.
- **Test overlap:** 3 of the 15,534 official test texts also appear in train/val. They are short, generic phrases (`انسرقت بطاقتي` in Saudi, `تفعيل بطاقتي` twice in Moroccan) written for different source questions. The test files stay as published, so our numbers compare with the paper.
- **Caveat:** the smallest intents have only 4 validation questions, so their per-intent F1 on validation is noisy. Compare models on macro-F1, not on single small intents.